# 03 – Character Nicknames

Esplorazione e data cleaning del dataset `character_nicknames.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `character_mal_id` | Chiave esterna che fa riferimento alla chiave primaria di `characters.csv` |
| `nickname` | Soprannome del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_nicknames = pd.read_csv('../datasets/character_nicknames.csv')
print(f'Shape: {df_nicknames.shape}')
df_nicknames.info()
df_nicknames.head()

Shape: (37080, 2)
<class 'pandas.DataFrame'>
RangeIndex: 37080 entries, 0 to 37079
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   character_mal_id  37080 non-null  int64
 1   nickname          37064 non-null  str  
dtypes: int64(1), str(1)
memory usage: 579.5 KB


,character_mal_id,nickname
0,280205,Hikaruko
1,280129,Hinacchi
2,280127,Bertha Willis
3,280066,Jimmy
4,280059,Full Body Red Square


**Osservazioni iniziali:**
Il dataset contiene **37.080 righe** e **2 colonne**.
`character_mal_id` è completo: **nessun valore nullo**
`nickname` presenta **16 valori nulli** (37.064 non-null su 37.080), da gestire in fase di cleaning.
I tipi di dati sono adeguati: `int64` e `str`

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_nicknames)

mask_dup = df_nicknames.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_nicknames[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_nicknames.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_nicknames):,}')

Righe totali coinvolte in duplicazioni : 168
  → prime occorrenze mantenute         : 66
  → occorrenze extra rimosse           : 102

Righe prima della rimozione : 37,080
Righe dopo la rimozione     : 36,978


**Osservazioni – duplicati esatti:**
**168 righe** (0.45% del totale) risultano coinvolte in duplicazioni esatte su entrambe le colonne. Di queste, **66** sono prime occorrenze mantenute e **102** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 37.080 a **36.978 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `characters.csv`. Per una chiave esterna le statistiche descrittive non hanno significato interpretativo.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `characters_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

I valori duplicati in questo caso sono attesi in quanto lo stesso personaggio può avere più di un alias. 

In [3]:
df_characters = pd.read_csv('../datasets_cleaned/characters_clean.csv')

mask_orphan = check_fk(
    df_nicknames['character_mal_id'],
    df_characters['character_mal_id'],
    child_df=df_nicknames
)

print(f'Null in character_mal_id  : {df_nicknames["character_mal_id"].isna().sum()}')
print(f'Duplicati in character_mal_id (attesi) : {df_nicknames["character_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        character_mal_id
  Colonna PK  (tabella padre)         character_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       36,978
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            36978  (100.00%)
  Valori unici nella PK padre         209,506

  ✓  Righe con FK valida              36978  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in character_mal_id  : 0
Duplicati in character_mal_id (attesi) : 6,696


**Osservazioni:**
- **Nessun valore nullo**: tutti i 36.978 record hanno un ID di riferimento.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia è necessaria.

### 2.2 `nickname`

È l'unica colonna con contenuto semantico del dataset: contiene il soprannome del personaggio in forma testuale libera.

A differenza di `character_mal_id`, qui ci interessa capire la distribuzione delle lunghezze, la presenza di valori nulli, duplicati, caratteri anomali, etc. Per questo utilizziamo `analyze`. 

I duplicati sono **attesi** in quanto soprannomi generici possono comparire per personaggi diversi e non costituiscono un problema.

In [4]:
analyze(df_nicknames['nickname'])


  Nome serie                     nickname
  dtype                          str
  Dimensione                     36,978
  Range indice                   0 … 37079
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   36,978
  Valori non nulli               36,962  (99.96%)
  Null / NaN                     16  (0.04%)
  Stringhe vuote                 0
  Valori unici                   28,928  (78.23%)
  Valori duplicati               8,034

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  87  caratteri
  Lunghezza media                9.84  caratteri
  Lunghezza mediana              8.0000  caratteri
  Dev. standard lunghezza        5.78
  IQR lunghezza                  7.0000

Valori di lunghezza estrema
─────────────────

  Contiene cifre                 661  (1.8%)
  Contiene maiuscole             36,733  (99.4%)
  Contiene minuscole             36,434  (98.6%)
  Contiene spazi                 16,473  (44.6%)
  Contiene caratteri speciali    4,562  (12.3%)
  Contiene URL                   0  (0.0%)
  Contiene @                     4  (0.0%)
  Solo numerico                  54
  Solo alfabetico                17,478
  Solo alfanumerico              17,605

Statistiche a livello di carattere
────────────────────────────────────────────────────────────────────────────────

  Caratteri totali               363,573
  Caratteri unici                399
  Caratteri medi per valore      9.84

Analisi parole / token
────────────────────────────────────────────────────────────────────────────────

  Parole medie per valore        1.65
  Max parole                     15
  Min parole                     0

  Le 10 parole più comuni:
  Parola                  Conteggio
  ────────────────────   ──────────
  of     

**Osservazioni:**
- Ci sono **16 valori nulli** che sono righe senza nickname prive di significato e sono da rimuovere nella sezione 2.3.
- Ci sono **28.928 valori unici** (78.23%) e i nickname duplicati (21.77%) sono attesi quindi non c'è nessuna anomalia.
- Lunghezza variabile da **1 carattere** a **87 caratteri**. Media: 9.84 caratteri.
- Il 57.4% dei nickname ha lunghezza tra 1 e 9 caratteri, confermando la prevalenza di soprannomi brevi.
- **54 nickname puramente numerici** che per il momento non presentano una anomalia ma sono da tenere presenti per analisi future.
- **12.3% contiene caratteri speciali**. Analizziamo più in profondità le seguenti anomalie:
  - **4 nickname contengono `@`**: da verificare (possibili typo).
  - **65 nickname contengono `;`**: probabili separatori tra alias multipli, da spezzare su righe distinte.
  - **Altri caratteri** (`/`, `(`, `#`, `&`): da verificare se sono separatori o parte legittima del testo.

##### Nickname con `@`

Quattro nickname contengono il carattere `@`, non atteso in un soprannome testuale. Potrebbe trattarsi di un typo. Visualizziamo le righe per decidere la correzione.

In [5]:
mask_at = df_nicknames['nickname'].str.contains('@', regex=False, na=False)
print(f'Righe con @ nel nickname: {mask_at.sum()}')
df_nicknames[mask_at]

Righe con @ nel nickname: 4


,character_mal_id,nickname
14077,150363,@NyanNyan
14078,150363,@Meow-Meow
14687,144978,S@ik1 KAzU
30570,18138,Riko @ Androids


Si nota che sono tutti da correggere. Ci sono 3 casi dove si tratta di caratteri extra da rimuovere, il un caso dove il carattere `@` è una sostituzione della lettera `a`. Correggiamo gli errori di battitura. 

In [6]:
corrections = {
    '@NyanNyan':       'NyanNyan',
    '@Meow-Meow':      'Meow-Meow',
    'S@ik1 KAzU':      'Saik1 KAzU',
    'Riko @ Androids': 'Riko Androids',
}

df_nicknames['nickname'] = df_nicknames['nickname'].replace(corrections)

remaining = df_nicknames['nickname'].str.contains('@', regex=False, na=False).sum()
print(f'Nickname con @ dopo la correzione: {remaining}')

Nickname con @ dopo la correzione: 0


##### Nickname con `;`

Il carattere `;` compare in alcune righe come separatore di alias multipli nello stesso campo. Queste righe andrebbero spezzate. Verifichiamo quante sono e come si presentano.

In [7]:
mask_semi = df_nicknames['nickname'].str.contains(';', regex=False, na=False)
print(f'Righe con ; nel nickname: {mask_semi.sum()}')
df_nicknames[mask_semi].head(10)

Righe con ; nel nickname: 65


,character_mal_id,nickname
2336,254754,Master; Sayer
2356,254664,Mantine's Trainer; Steamboat's Owner; Old Man
2408,254205,Masked Man; 仮面の男
2483,253413,Poppuru; Populu
4052,238387,Ken; Ken-nii
5525,224128,Qing Feng; Green Phoenix
5808,221633,Shark; Ball Hunter
5918,220321,Pepe the Frog; Shangxin Qingwa
7829,202827,Leo; Demon King Leonis
9011,193852,Fabius Maximus Verruscosus; Cunctator


Le righe con `;` contengono più alias nello stesso campo. Poiché il dataset è strutturato con un nickname per riga, le separiamo mantenendo lo stesso `character_mal_id`.

In [8]:
n_prima = len(df_nicknames)

df_nicknames['nickname'] = df_nicknames['nickname'].str.split(';')
df_nicknames = df_nicknames.explode('nickname').reset_index(drop=True)
df_nicknames['nickname'] = df_nicknames['nickname'].str.strip()

# Rimuoviamo eventuali stringhe vuote generate da ; finali
df_nicknames = df_nicknames[df_nicknames['nickname'] != ''].reset_index(drop=True)

n_dopo = len(df_nicknames)
print(f'Righe prima della separazione : {n_prima:,}')
print(f'Righe dopo la separazione     : {n_dopo:,}')
print(f'Nuove righe generate           : {n_dopo - n_prima:,}')

Righe prima della separazione : 36,978
Righe dopo la separazione     : 37,057
Nuove righe generate           : 79


##### Altri caratteri speciali (`/`, `(`, `#`, `&`)

`analyze` ha rilevato che il 12.3% dei nickname contiene caratteri speciali. Dopo aver gestito `;` e `@`, verifichiamo gli altri caratteri potenzialmente anomali per determinare se sono separatori o testo legittimo. Analiziamo anche i nickname interamente numerici. 

In [16]:
# Conteggio per carattere speciale (esclusi @ e ; già gestiti)
conteggi = (
    df_nicknames['nickname']
    .str.extractall(r'([/(#&!])')
    .rename(columns={0: 'carattere'})
    ['carattere']
    .value_counts()
    .sort_index()
    .rename('occorrenze')
    .rename_axis('carattere')
)
print(conteggi.to_string())
print()

# Top 5 esempi per carattere
esempi = (
    df_nicknames
    .assign(carattere=df_nicknames['nickname'].str.extract(r'([/(#&!])'))
    .dropna(subset=['carattere'])
    .groupby('carattere')['nickname']
    .value_counts()
    .groupby(level=0)
    .head(5)
)
print(esempi.to_string())

mask_num = df_nicknames['nickname'].str.fullmatch(r'\d+', na=False)
print(f'Nickname solo numerici: {mask_num.sum()}')
print()
print(df_nicknames[mask_num][['character_mal_id', 'nickname']].to_string())

carattere
!     1
#    53
&     9
(    78
/    32

carattere  nickname                    
!          Take!                           1
#          Girl #12                        2
           Coal Miner #3                   1
           #6                              1
           Sukedachi #2                    1
           #2                              1
&          Goddess of Love & Beauty        1
           Blake & Heath's Mother          1
           Blake & Heath's Father          1
           Black & White                   1
           Heckler & Koch Gewehr 36        1
(          The Twins (Soushikyuu)          2
           God of Emotions(Former)         2
           The Paired Fish (Sougyokyuu)    2
           Niku (Meat)                     1
           Anthony (Hoenn)                 1
/          Chef Jack/John                  1
           Michael/Michel                  1
           AR-6/GR-6                       1
           Narrator / Protagonist          1
         

**Osservazioni:**
- **`/`** (32 righe): separatore tra varianti del nome (es. `Beast III/L`, `Kama/Mara`). Non è un separatore di alias distinti ma parte integrante del soprannome. **Nessuna pulizia necessaria.**
- **`(`** (78 righe): traslitterazione o chiarimento tra parentesi (es. `The Paired Fish (Sougyokyuu)`). Fanno parte del nickname. **Nessuna pulizia necessaria.**
- **`#`** (54 righe): designazioni numeriche (es. `Girl #12`, `Perman #3`). Eccezione: 1 riga contiene `&#0`, un'entità HTML non decodificata da correggere. **Pulizia parziale necessaria.**
- **`&`** (10 righe): usato come congiunzione (es. `Heaven & Hell`). Eccezione: la stessa riga con `&#0` sopra. **Nessuna ulteriore pulizia necessaria.**
- **`!`** (1 riga): `Take!` — nome con esclamativo, valido. **Nessuna pulizia necessaria.**
- **solo numerici** (54 nickname): Sono designazioni numeriche ufficiali (es. codici clone, numeri di unità), coerenti con quanto già osservato in `characters_clean.csv`. **Nessuna pulizia necessaria.**

##### HTML entity `&#0`

Una riga contiene `Red Dragon Emperor&#0`: si tratta di un'entità HTML numerica non decodificata (`&#0` = carattere nullo Unicode). Il valore corretto è `Red Dragon Emperor`.

In [10]:
mask_html = df_nicknames['nickname'].str.contains('&#', regex=False, na=False)
print(f'Righe con HTML entity: {mask_html.sum()}')
print(df_nicknames[mask_html][['character_mal_id', 'nickname']].to_string())

import re
df_nicknames['nickname'] = df_nicknames['nickname'].str.replace(r'&#\d+', '', regex=True).str.strip()

print()
print('Dopo la correzione:')
print(df_nicknames.loc[mask_html[mask_html].index, ['character_mal_id', 'nickname']].to_string())

Righe con HTML entity: 1
      character_mal_id               nickname
6886            211142  Red Dragon Emperor&#0

Dopo la correzione:
      character_mal_id            nickname
6886            211142  Red Dragon Emperor


### 2.3 Chiave primaria `(character_mal_id, nickname)`

Dopo tutte le operazioni di pulizia, verifichiamo che la coppia `(character_mal_id, nickname)` sia univoca: un personaggio non dovrebbe avere lo stesso soprannome registrato due volte.

Infine rimuoviamo le 16 righe con `nickname` nullo rimaste dal dataset originale.

In [12]:
# Verifica PK (character_mal_id, nickname)
n_pk_dup = df_nicknames.duplicated(subset=['character_mal_id', 'nickname']).sum()
print(f'Duplicati su (character_mal_id, nickname): {n_pk_dup}')
if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_nicknames.drop_duplicates(subset=['character_mal_id', 'nickname'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_nicknames):,}')
else:
    print('→ Chiave primaria già univoca.')
print()

# Rimozione null
print(f'Null in nickname prima della pulizia: {df_nicknames["nickname"].isna().sum()}')
df_nicknames.dropna(subset=['nickname'], inplace=True)
print(f'Null in nickname dopo la pulizia    : {df_nicknames["nickname"].isna().sum()}')
print(f'Righe dopo pulizia nickname         : {len(df_nicknames):,}')

Duplicati su (character_mal_id, nickname): 0
→ Chiave primaria già univoca.

Null in nickname prima della pulizia: 16
Null in nickname dopo la pulizia    : 0
Righe dopo pulizia nickname         : 37,041


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [13]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali               : {n_originale:>10,}')
print(f'  - duplicate esatte rimosse   : {-102:>10,}')
print(f'  + righe generate da split ;  : {+79:>10,}')
print(f'  - nickname nulli rimossi     : {-16:>10,}')
print(f'  - HTML entity corretta       : {0:>10,}  (correzione, non rimozione)')
print(f'Righe dopo cleaning           : {len(df_nicknames):>10,}')
print(f'Saldo netto                   : {len(df_nicknames) - n_originale:>+10,}')
print()
print('Dtypes finali:')
print(df_nicknames.dtypes)
print()
print('Valori mancanti residui:')
print(df_nicknames.isnull().sum())
print()
df_nicknames.to_csv('../datasets_cleaned/character_nicknames_clean.csv', index=False)
print('Salvato: datasets_cleaned/character_nicknames_clean.csv')

Riepilogo Dataset Pulito
Righe originali               :     37,080
  - duplicate esatte rimosse   :       -102
  + righe generate da split ;  :         79
  - nickname nulli rimossi     :        -16
  - HTML entity corretta       :          0  (correzione, non rimozione)
Righe dopo cleaning           :     37,041
Saldo netto                   :        -39

Dtypes finali:
character_mal_id    int64
nickname              str
dtype: object

Valori mancanti residui:
character_mal_id    0
nickname            0
dtype: int64



Salvato: datasets_cleaned/character_nicknames_clean.csv
